# AR Parameters


In [1]:
import sys
from pathlib import Path

# resolve project root (two levels up from this notebook)
PROJECT_ROOT = Path.cwd().resolve().parents[1]

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

for p in (SRC_DIR, NOTEBOOKS_DIR):
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

In [2]:
import numpy as np
import yaml

from parameterization.utils.storage import load_ar_p_parameters
from notebook_utils import generate_sweep_dict_list
from utils.config import ConfigFlowTraining, ConfigParamsFit
from utils.sweep_utils import get_sweep_name

In [3]:
results_dir = PROJECT_ROOT / "results"
data_dirs = [
    results_dir / "fitted_parameters",
    results_dir / "parameters_flow",
    results_dir / "parameters_forcing_flow",
    results_dir / "parameters_history_flow",
    results_dir / "parameters_tail_flow",
]

ar_base_flow_dir = results_dir / "parameters_ar_base_flow"

data_dirs_tv = [
    results_dir / "fitted_parameters_time_variant_forcing",
    results_dir / "parameters_flow_time_variant_forcing",
    results_dir / "parameters_forcing_flow_time_variant_forcing",
    results_dir / "parameters_history_flow_time_variant_forcing",
    results_dir / "parameters_tail_flow_time_variant_forcing",
]


In [4]:
def load_yaml(file_path):
    with open(file_path, "r") as f:
        return yaml.safe_load(f)

In [5]:
DEFAULT_SWEEP_KEY = "default"


def load_ar_parameters(data_dirs):
    ar_params_dict = {}
    for d in data_dirs:
        print(f"Loading rho, sigma from {d.name}...")

        sweep_path = d / "sweep.yaml"
        sweep = load_yaml(sweep_path)

        ar_params_dict[d.name] = {"sweep": sweep}

        for s in generate_sweep_dict_list(sweep):
            sweep_name = get_sweep_name(s)
            out_path = d / sweep_name

            sweep_key = sweep_name if sweep_name != "" else DEFAULT_SWEEP_KEY

            if "flow" in d.name:
                config_path = out_path / "config.yaml"
                config = ConfigFlowTraining(config_path)
            else:
                # bayes fitting can overwrite config.yaml, so we use the base one
                config_path = d / "fit_params_baseline.yaml"
                config = ConfigParamsFit(config_path)

            ar_orders = config.ar_order
            if not isinstance(ar_orders, list):
                ar_orders = [ar_orders]

            ar_params_dict[d.name][sweep_key] = {p: {} for p in ar_orders}

            for p in ar_orders:
                rho, sigma = load_ar_p_parameters(
                    config.ar_parameters_dir(out_path), ar_order=p
                )

                ar_params_dict[d.name][sweep_key][p] = {"rho": rho, "sigma": sigma}

    print("Done.")
    return ar_params_dict

In [6]:
ar_params_dict = load_ar_parameters(data_dirs)

Loading rho, sigma from fitted_parameters...
Loading rho, sigma from parameters_flow...
Loading rho, sigma from parameters_forcing_flow...
Loading rho, sigma from parameters_history_flow...
Loading rho, sigma from parameters_tail_flow...
Done.


In [7]:
def print_ar_parameters(ar_params_dict):
    for k, v in ar_params_dict.items():
        print(f"{k}: ")
        for sweep_key, ar_params in v.items():
            if sweep_key == "sweep":
                continue
            print(f"  {sweep_key}:")
            for p, params in ar_params.items():
                rho, sigma = params["rho"], params["sigma"]
                sigma_e = sigma / np.sqrt(1 - rho**2)
                if isinstance(rho, float):
                    print(
                        f"    AR order {p}: rho={rho:.4f}, sigma={sigma:.4f}, sigma_e={sigma_e:.4f}"
                    )
                else:
                    print(f"    AR order {p}: rho={rho}, sigma={sigma:.4f}")

In [8]:
print_ar_parameters(ar_params_dict)

fitted_parameters: 
  default:
    AR order 1: rho=0.9856, sigma=0.3382, sigma_e=1.9981
parameters_flow: 
  default:
    AR order 1: rho=0.9758, sigma=0.2199, sigma_e=1.0051
parameters_forcing_flow: 
  default:
    AR order 1: rho=0.9755, sigma=0.2200, sigma_e=1.0012
parameters_history_flow: 
  delta_t_1:
    AR order 1: rho=0.9759, sigma=0.2119, sigma_e=0.9705
  delta_t_2:
    AR order 1: rho=0.9760, sigma=0.2127, sigma_e=0.9762
parameters_tail_flow: 
  default:
    AR order 1: rho=0.9733, sigma=0.2224, sigma_e=0.9686


In [9]:
# Loading AR parameters for the base flow separately,
# since it fitted after the others and doesn't have an AR order list
DEFAULT_SWEEP_KEY = "default"

print(f"Loading rho, sigma from {ar_base_flow_dir.name}...")

sweep_path = ar_base_flow_dir / "sweep.yaml"
sweep = load_yaml(sweep_path)

ar_params_dict[ar_base_flow_dir.name] = {"sweep": sweep}

for s in generate_sweep_dict_list(sweep):
    sweep_name = get_sweep_name(s)
    out_path = ar_base_flow_dir / sweep_name

    sweep_key = sweep_name if sweep_name != "" else DEFAULT_SWEEP_KEY

    config_path = out_path / "config.yaml"
    config = ConfigFlowTraining(config_path)

    p = config.ar_order

    rho, sigma = load_ar_p_parameters(config.ar_parameters_dir(out_path), ar_order=p)

    ar_params_dict[ar_base_flow_dir.name][sweep_key] = {p: {"rho": rho, "sigma": sigma}}

print("Done.")


Loading rho, sigma from parameters_ar_base_flow...
Done.


In [10]:
print_ar_parameters(ar_params_dict)

fitted_parameters: 
  default:
    AR order 1: rho=0.9856, sigma=0.3382, sigma_e=1.9981
parameters_flow: 
  default:
    AR order 1: rho=0.9758, sigma=0.2199, sigma_e=1.0051
parameters_forcing_flow: 
  default:
    AR order 1: rho=0.9755, sigma=0.2200, sigma_e=1.0012
parameters_history_flow: 
  delta_t_1:
    AR order 1: rho=0.9759, sigma=0.2119, sigma_e=0.9705
  delta_t_2:
    AR order 1: rho=0.9760, sigma=0.2127, sigma_e=0.9762
parameters_tail_flow: 
  default:
    AR order 1: rho=0.9733, sigma=0.2224, sigma_e=0.9686
parameters_ar_base_flow: 
  default:
    AR order 1: rho=0.9926, sigma=0.1133, sigma_e=0.9349
